# Pipeline de Entrenamiento en Google Colab — Norma ERP (Ecuador)

Este cuaderno entrena y exporta los artefactos de Inteligencia Artificial de **Norma ERP** a partir de la extracción íntegra (100%) de los 4 cuerpos normativos oficiales y las síntesis doctrinales (1,895 fragmentos indexados):
1. **Constitución de la República del Ecuador (2008)** — 176 páginas (481 artículos/secciones).
2. **Código del Trabajo del Ecuador** — 185 páginas (714 artículos/secciones).
3. **Ley de Seguridad Social (IESS)** — 112 páginas (383 artículos/secciones).
4. **Manual Metodológico OIT — Indicadores de Trabajo Decente (ODS 8)** — 67 páginas (302 secciones).
5. **Síntesis Doctrinales y Fórmulas Deterministas (`Decimal.js`)** — 15 compendios verificados.

---

## Archivos que debes subir a Google Colab (Panel izquierdo -> Archivos)
Desde la carpeta `docs/09-sostenibilidad-ods/` de tu proyecto, sube estos 2 archivos:
1. `retriever_evaluator_training_colab.jsonl` *(1,880 tripletas contrastivas: `query` + `positive` + `hard_negative`)*
2. `fine_tuning_ecuador_ods8_qwen.jsonl` *(275 muestras ChatML multi-dominio con cálculos exactos `Decimal.js` y muestreo estratificado de los 4 PDFs)*

## Qué produce este cuaderno
- **Fase 1 (Entrenamiento del Modelo Buscador Bi-Encoder + Modelo Evaluador Cross-Encoder):**
  - Entrena el **Modelo Buscador Bi-Encoder (`intfloat/multilingual-e5-base`)** con `MultipleNegativesRankingLoss` sobre los 1,895 fragmentos.
  - Entrena el **Modelo Evaluador Cross-Encoder (`unicamp-dl/mMiniLM-L6-v2-mmarco-v2`)** para re-ranking de precisión legal y detección de alucinaciones.
  - Genera y descarga automáticamente el archivo unificado **`trained_retriever_index.json`** (con los vectores densos de los 1,895 artículos, los puntajes calibrados del Cross-Encoder y los vectores del vocabulario de consulta), listo para copiarse en:
    `backend/src/modules/sustainability/data/trained_retriever_index.json`
- **Fase 2 (Fine-Tuning QLoRA 4-bit del LLM Generativo con Unsloth):**
  - Ejecuta Fine-Tuning Supervisado (SFT + QLoRA 4-bit) sobre **`Qwen/Qwen2.5-7B-Instruct`** (compatible con GPU T4 gratuita de Colab) o **`Qwen/Qwen2.5-14B-Instruct`** (GPU A100/L4).
  - Evalúa el modelo en vivo dentro de Colab, exporta el archivo cuantizado **`.gguf` (`q4_k_m`)** y genera el **`Modelfile`** listo para servir con **Ollama** (`AI_PROVIDER=local_ollama`) o **LM Studio / vLLM** (`AI_PROVIDER=local_openai`).

In [ ]:
# ==============================================================================
# PASO 1: INSTALACIÓN DE DEPENDENCIAS PARA EL BUSCADOR Y EVALUADOR (FASE 1)
# ==============================================================================
!pip install -q "sentence-transformers>=3.0.0" "datasets>=2.19.0" "accelerate>=0.30.0" scikit-learn

In [ ]:
# ==============================================================================
# PASO 2: VERIFICACIÓN DE LOS DATASETS SUBIDOS A COLAB
# ==============================================================================
import os
import re
import json
import unicodedata
from collections import Counter

RETRIEVER_DATASET_PATH = "retriever_evaluator_training_colab.jsonl"
SFT_DATASET_PATH = "fine_tuning_ecuador_ods8_qwen.jsonl"

if not os.path.exists(RETRIEVER_DATASET_PATH):
    raise FileNotFoundError(
        f"No se encontró {RETRIEVER_DATASET_PATH}. Súbelo desde docs/09-sostenibilidad-ods/ al panel izquierdo de Archivos en Colab."
    )

with open(RETRIEVER_DATASET_PATH, "r", encoding="utf-8") as f:
    retriever_rows = [json.loads(line) for line in f if line.strip()]

print(f"[OK] Dataset de Buscador/Evaluador cargado: {len(retriever_rows)} fragmentos legales.")
if os.path.exists(SFT_DATASET_PATH):
    with open(SFT_DATASET_PATH, "r", encoding="utf-8") as f:
        sft_rows = [json.loads(line) for line in f if line.strip()]
    print(f"[OK] Dataset SFT/QLoRA cargado: {len(sft_rows)} muestras ChatML multi-dominio.")
else:
    print(f"[INFO] {SFT_DATASET_PATH} aún no subido (solo requerido si ejecutarás la Fase 2).")

In [ ]:
# ==============================================================================
# PASO 3: ENTRENAMIENTO DEL MODELO BUSCADOR (BI-ENCODER RETRIEVER)
# Entrena el buscador semántico con los 1,895 artículos de los 4 PDFs oficiales
# ==============================================================================
import torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

BI_ENCODER_MODEL_NAME = "intfloat/multilingual-e5-base"
OUTPUT_RETRIEVER_DIR = "./norma_erp_legal_retriever"

print(f"Cargando modelo base Bi-Encoder: {BI_ENCODER_MODEL_NAME}...")
retriever_model = SentenceTransformer(BI_ENCODER_MODEL_NAME)

# Construcción de tripletas contrastivas (query, positive_passage, hard_negative_passage)
train_examples = []
for row in retriever_rows:
    query_text = f"query: {row['query']}"
    pos_text = f"passage: {row['article_code']} - {row['positive']}"
    neg_text = f"passage: {row['hard_negative']}"
    train_examples.append(InputExample(texts=[query_text, pos_text, neg_text]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model=retriever_model)

print(f"Iniciando entrenamiento del Modelo Buscador sobre {len(train_examples)} artículos...")
retriever_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=int(len(train_dataloader) * 0.1),
    output_path=OUTPUT_RETRIEVER_DIR,
    show_progress_bar=True,
)
print(f"[OK] Modelo Buscador entrenado y guardado en: {OUTPUT_RETRIEVER_DIR}")

In [ ]:
# ==============================================================================
# PASO 4: ENTRENAMIENTO DEL MODELO EVALUADOR (CROSS-ENCODER RERANKER)
# Califica relevancia exacta [0.0 a 1.0] entre la consulta y el artículo legal
# ==============================================================================
import numpy as np
from sentence_transformers.cross_encoder import CrossEncoder

CROSS_ENCODER_MODEL_NAME = "unicamp-dl/mMiniLM-L6-v2-mmarco-v2"
OUTPUT_EVALUATOR_DIR = "./norma_erp_legal_evaluator"

print(f"Cargando Cross-Encoder Evaluador base: {CROSS_ENCODER_MODEL_NAME}...")
evaluator_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, num_labels=1, max_length=512)

ce_train_samples = []
for row in retriever_rows:
    ce_train_samples.append(
        InputExample(texts=[row["query"][:250], f"{row['article_code']}: {row['positive'][:1000]}"], label=1.0)
    )
    ce_train_samples.append(
        InputExample(texts=[row["query"][:250], row["hard_negative"][:1000]], label=0.0)
    )

ce_dataloader = DataLoader(ce_train_samples, shuffle=True, batch_size=16)

print(f"Entrenando Modelo Evaluador Cross-Encoder con {len(ce_train_samples)} pares...")
evaluator_model.fit(
    train_dataloader=ce_dataloader,
    epochs=1,
    warmup_steps=int(len(ce_dataloader) * 0.1),
    output_path=OUTPUT_EVALUATOR_DIR,
    show_progress_bar=True,
)

# Calcular el score calibrado [0.0, 1.0] de cada artículo con el Cross-Encoder entrenado
positive_pairs = [[r["query"], f"{r['article_code']}: {r['positive']}"] for r in retriever_rows]
raw_ce_logits = evaluator_model.predict(positive_pairs, batch_size=32, show_progress_bar=True)
calibrated_ce_scores = 1.0 / (1.0 + np.exp(-np.array(raw_ce_logits)))
print(f"[OK] Modelo Evaluador entrenado. Confianza media sobre positivos: {calibrated_ce_scores.mean():.4f}")

In [ ]:
# ==============================================================================
# PASO 5: EXPORTACIÓN UNIFICADA DEL ÍNDICE NEURONAL (trained_retriever_index.json)
# Incluye: embeddings de los 1,895 artículos + scores del Cross-Encoder +
# embeddings del vocabulario jurídico/contable para inferencia híbrida en Node.js (<3ms)
# ==============================================================================
EMBED_DIM = 128

def normalize_token_es(text):
    s = unicodedata.normalize("NFD", str(text or "").lower())
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z0-9.%]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def project_and_l2_normalize(vecs, dim=EMBED_DIM):
    sliced = np.array(vecs, dtype=np.float32)[:, :dim]
    norms = np.linalg.norm(sliced, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return sliced / norms

print("1/2 Codificando los 1,895 artículos legales con el Bi-Encoder entrenado...")
passages = [f"passage: {r['article_code']} - {r['positive']}" for r in retriever_rows]
full_doc_embeddings = retriever_model.encode(
    passages,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
doc_embeddings_128 = project_and_l2_normalize(full_doc_embeddings, EMBED_DIM)

# Extraer los 800 términos jurídicos, contables y laborales más relevantes para el índice de consulta
token_counter = Counter()
for r in retriever_rows:
    tokens = [t for t in normalize_token_es(f"{r['article_code']} {r['query']} {r['positive'][:400]}").split() if len(t) >= 3]
    token_counter.update(tokens)

top_vocab_terms = [term for term, _ in token_counter.most_common(800)]
print(f"2/2 Codificando {len(top_vocab_terms)} términos del vocabulario de consulta con el Bi-Encoder...")
vocab_queries = [f"query: {term}" for term in top_vocab_terms]
full_vocab_embeddings = retriever_model.encode(
    vocab_queries,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)
vocab_embeddings_128 = project_and_l2_normalize(full_vocab_embeddings, EMBED_DIM)

query_vocabulary_embeddings = {
    term: [round(float(x), 4) for x in vocab_embeddings_128[i]]
    for i, term in enumerate(top_vocab_terms)
}

exported_articles = []
for idx, row in enumerate(retriever_rows):
    exported_articles.append({
        "id": idx,
        "articleCode": row["article_code"],
        "title": row.get("title", ""),
        "category": row.get("category", "LABOR_CODE"),
        "odsTarget": row.get("ods_target", "8.8"),
        "domain": row.get("domain", "LABOR_CODE"),
        "crossEncoderScore": round(float(calibrated_ce_scores[idx]), 4),
        "embedding": [round(float(x), 4) for x in doc_embeddings_128[idx]],
    })

index_payload = {
    "model": f"{BI_ENCODER_MODEL_NAME} + {CROSS_ENCODER_MODEL_NAME}",
    "totalArticles": len(exported_articles),
    "embeddingDim": EMBED_DIM,
    "articles": exported_articles,
    "queryVocabularyEmbeddings": query_vocabulary_embeddings,
}

INDEX_FILENAME = "trained_retriever_index.json"
with open(INDEX_FILENAME, "w", encoding="utf-8") as f:
    json.dump(index_payload, f, ensure_ascii=False)

print(f"[OK] Índice exportado: {INDEX_FILENAME} ({len(exported_articles)} artículos + {len(query_vocabulary_embeddings)} vectores de consulta).")
print("Copia este archivo en: backend/src/modules/sustainability/data/trained_retriever_index.json")

try:
    from google.colab import files
    files.download(INDEX_FILENAME)
except Exception:
    print("Descarga manual disponible en el panel lateral de archivos de Colab.")

---
## Fase 2: Fine-Tuning QLoRA del Modelo Generativo Local (Qwen 2.5 / DeepSeek Distill)

- Si utilizas una API de frontera (**DeepSeek**, **OpenAI**, **Gemini**, **Groq** o **Anthropic**), el archivo `trained_retriever_index.json` generado en el Paso 5 ya conecta el Buscador y el Evaluador con tu API.
- Ejecuta los **Pasos 6, 7 y 8** para entrenar tu propio LLM especializado en legislación ecuatoriana, contabilidad NIIF y ODS 8, y exportarlo a `.gguf` (`q4_k_m`) para correrlo con **Ollama** o **LM Studio**.

In [ ]:
# ==============================================================================
# PASO 6: INSTALACIÓN DE UNSLOTH PARA FINE-TUNING QLORA (4-BIT)
# ==============================================================================
!pip install -q unsloth

In [ ]:
# ==============================================================================
# PASO 7: FINE-TUNING SUPERVISADO (SFT + QLoRA 4-BIT) CON EVALUACIÓN
# Modelos recomendados:
# - "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" (Ideal para GPU T4 gratuita de Colab)
# - "unsloth/Qwen2.5-14B-Instruct-bnb-4bit" (Ideal para GPU A100 / L4)
# - "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit"
# ==============================================================================
import sys
sys.modules["torchao"] = None  # Evita conflicto con torchao preinstalado en Colab
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

raw_dataset = load_dataset("json", data_files=SFT_DATASET_PATH, split="train")

def format_chatml(batch):
    texts = []
    for msgs in batch["messages"]:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"formatted_chat": texts}

dataset = raw_dataset.map(format_chatml, batched=True)
split_dataset = dataset.train_test_split(test_size=0.1, seed=3407)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    dataset_text_field="formatted_chat",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs_norma_erp",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"[OK] Entrenamiento QLoRA finalizado. Loss final: {trainer_stats.training_loss:.4f}")

In [ ]:
# ==============================================================================
# PASO 8: PRUEBA EN VIVO DEL MODELO ENTRENADO Y EXPORTACIÓN A GGUF + MODELFILE
# ==============================================================================
FastLanguageModel.for_inference(model)

test_messages = [
    {
        "role": "system",
        "content": "Eres el Asistente Experto en Legislación Laboral del Ecuador, Contabilidad Financiera NIIF para PYMES, Tributación SRI, Seguridad Social IESS y Gobernanza ODS 8 integrado en Norma ERP.",
    },
    {
        "role": "user",
        "content": "Calcula paso a paso la indemnización por despido intempestivo (Art. 188) y bonificación por desahucio (Art. 185) en Ecuador para un trabajador con sueldo mensual de $850.00 USD y 4 años y 2 meses de servicio.",
    },
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1, use_cache=True)
response_text = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print("=== RESPUESTA DE PRUEBA DEL MODELO FINE-TUNEADO ===")
print(response_text)

# Exportar a formato GGUF cuantizado (q4_k_m) para Ollama / LM Studio
GGUF_DIR = "norma_erp_ecuador_gguf"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

# Generar Modelfile oficial para importar en Ollama con: ollama create ecuador-erp-ai -f Modelfile
modelfile_content = '''FROM ./unsloth.Q4_K_M.gguf
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
SYSTEM """Eres el Asistente Experto en Legislación Laboral del Ecuador, Contabilidad Financiera NIIF para PYMES, Tributación SRI, Seguridad Social IESS, Protección de Datos (LOPDP) y Gobernanza ODS 8 integrado en Norma ERP. Responde siempre con rigor jurídico, precisión matemática de 2 decimales (ROUND_HALF_UP) y referencias exactas a la normativa ecuatoriana vigente."""
'''
with open(os.path.join(GGUF_DIR, "Modelfile"), "w", encoding="utf-8") as mf:
    mf.write(modelfile_content)

print(f"[OK] Modelo GGUF y Modelfile exportados en: {GGUF_DIR}")